# Neurocontroller – Visualize Existing Results

This notebook loads previously simulated trials and reproduces the standard analysis plots.
It is intended for the EBRAINS Software Distribution (ESD) and assumes the `runs/` directory
contains completed simulations.

In [ ]:
from pathlib import Path
from IPython.display import display, Image as IPImage

%matplotlib inline

from neurocontroller.config.paths import RUNS_DIR, RunPaths
from neurocontroller.config.ResultMeta import ResultMeta
from neurocontroller.plant.plant_plotting import plot_plant_outputs
from neurocontroller.neural.plot_utils import plot_controller_outputs
from neurocontroller.utils_common.draw_schema_svg import draw_schema
from neurocontroller.utils_common.results import gather_metas
from neurocontroller.plot_from_run import find_most_recent_run

## 1. Select a run

You can either:
- leave `RUN_ID` empty to auto-select the most recent run, or
- paste a specific run ID (the timestamp prefix is enough).

In [ ]:
RUN_ID = ""  # <-- paste a run ID here, or leave empty for latest

if not RUN_ID:
    recent = find_most_recent_run()
    if recent is None:
        raise FileNotFoundError(f"No runs found in {RUNS_DIR}")
    RUN_ID = recent.name
    print(f"Auto-selected most recent run: {RUN_ID}")
else:
    print(f"Using provided run ID: {RUN_ID}")

## 2. Load the trial chain

Trials can be chained (each new trial uses the previous as `parent_id`).
`gather_metas()` walks the parent chain and returns all trials in chronological order.

In [ ]:
metas = list(reversed(gather_metas(RUN_ID)))
print(f"Loaded {len(metas)} trial(s)")
for m in metas:
    print(f"  - {m.id}  (parent: {m.parent or 'none'})")

## 3. Plant-side analysis

Joint trajectories, motor commands, RMSE over trials, and end-effector error.

In [ ]:
plot_plant_outputs(metas)
draw_schema(metas)

In [ ]:
# Display plant figures from the *last* run in the chain
ref_paths = RunPaths.from_run_id(metas[-1].id, create_if_not_present=False)
plant_figs = sorted(ref_paths.figures_receiver.glob("*.png"))
print(f"Found {len(plant_figs)} plant figure(s)\n")
for fig_path in plant_figs:
    print(fig_path.name)
    display(IPImage(filename=str(fig_path)))

## 4. Neural-side analysis (optional)

Population rasters and PSTHs for planner, sensory, motor, and cerebellar populations.
> **Note:** This can be slow for long trial chains because it concatenates spike data.

In [ ]:
# Uncomment to generate neural plots
# plot_controller_outputs(metas)

# neural_figs = sorted(ref_paths.figures.glob("*.png"))
# print(f"Found {len(neural_figs)} neural figure(s)\n")
# for fig_path in neural_figs:
#     print(fig_path.name)
#     display(IPImage(filename=str(fig_path)))

## 5. Controller schema

A network diagram with embedded population plots.

In [ ]:
schema_png = ref_paths.figures_receiver / "whole_controller_schema.png"
if schema_png.exists():
    display(IPImage(filename=str(schema_png)))
else:
    print("Schema PNG not found.")

## 6. Quick data inspection

You can also load the raw Pydantic models to inspect numeric results programmatically.

In [ ]:
# Example: final error per trial
for m in metas:
    robot = m.load_robotic()
    print(f"{m.id}: final error = {robot.error[0]:.4f} rad")